In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="claude-haiku-4-5-20251001", #"gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="claude-haiku-4-5-20251001",  #"gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [5]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nUser is asking factual and speculative questions about the moon, its capital city Lunapolis, and the socio-economic conditions there.\n\n## SUMMARY\n\n- The moon's capital is Lunapolis\n- Weather in Lunapolis: clear skies, high of 120C, low of -100C\n- Population of cheese miners in Lunapolis: 100,000\n- Prediction: the cheese miners' union will likely strike due to dissatisfaction with the new president\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nUser may ask follow-up questions about Lunapolis, the cheese miners' union, the new president, or other lunar topics. Be prepared to provide additional information or analysis on these subjects.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='bec7b76f-485c-454b-95c8-737719f03bf2'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", ad

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [6]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

User is asking factual and speculative questions about the moon, its capital city Lunapolis, and the socio-economic conditions there.

## SUMMARY

- The moon's capital is Lunapolis
- Weather in Lunapolis: clear skies, high of 120C, low of -100C
- Population of cheese miners in Lunapolis: 100,000
- Prediction: the cheese miners' union will likely strike due to dissatisfaction with the new president

## ARTIFACTS

None

## NEXT STEPS

User may ask follow-up questions about Lunapolis, the cheese miners' union, the new president, or other lunar topics. Be prepared to provide additional information or analysis on these subjects.


## Trim/delete messages

In [7]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [8]:
agent = create_agent(
    model="claude-haiku-4-5-20251001", #gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [9]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='e237f639-5a15-4e7f-8948-313cef1c82d2'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='9955f793-dbd7-4d3b-a961-8df49a7a99e9', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8b9b72c7-d9cd-433a-ae46-1d3f525303e5'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0a3a6755-2385-4f19-ac44-89479ef02f71', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='16bd62e7-93c4-44d5-97e7-c5673cd3e93c'),
              AIMessage(content="Good question - temperature can be a clue. Check if the device

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [10]:
print(response["messages"][-1].content)

Good question - temperature can be a clue. Check if the device feels:

- **Normal/cool** - suggests it's not getting power
- **Warm/hot** - suggests it has power but may have an issue

What do you feel? Also, it would help to know:

1. **What device is it?** (phone, laptop, TV, etc.)
2. **Any lights or sounds?** (LEDs, fans, beeping)
3. **How long has it been plugged in?**

These details will help narrow down whether it's a power issue, battery problem, or something else.
